In [1]:
# Install & pull latest code:

import os

!pip install transformers accelerate -q

REPO_DIR = "/kaggle/working/vision-search-engine"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/3-pi-radians/vision-search-engine.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

Cloning into '/kaggle/working/vision-search-engine'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 41 (delta 15), reused 37 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 11.93 KiB | 2.98 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/kaggle/working/vision-search-engine
4e376e1 (HEAD -> main, origin/main, origin/HEAD) fixed the path in run_blip2_caption.py -> now poins to kaggle dataset instead of working directory
0f7e139 Updated config.py for
2bc618a Fixed the path mismatch in config.py


In [2]:
# Verifying inputs

import os

checks = {
    "image_paths.json": "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/image_paths.json",
    "crops/ folder":    "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops",
    "annotations":      "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-annotations/list_eval_partition.txt",
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    print(f"  {name}: {'✅' if exists else '❌ NOT FOUND'}")
    if not exists:
        all_ok = False

print("\n✅ Ready to run!" if all_ok else "\n❌ Fix missing inputs before running")

  image_paths.json: ✅
  crops/ folder: ✅
  annotations: ✅

✅ Ready to run!


In [3]:
# Verify config paths

!grep -n "IMAGE_PATHS_PATH\|CROPS_DIR\|CAPTIONS_PATH" /kaggle/working/vision-search-engine/config.py

28:CROPS_DIR        = _CROPS_DATASET / "crops"                if KAGGLE else WORK_DIR / "crops"
29:IMAGE_PATHS_PATH = _CROPS_DATASET / "image_paths.json"     if KAGGLE else WORK_DIR / "image_paths.json"
42:CAPTIONS_PATH      = WORK_DIR / "captions.json"


In [4]:
#  Run BLIP-2

!python offline/run_blip2_caption.py

2026-04-28 11:46:06,730 INFO Unique items: 3985 | Already captioned: 0 | Remaining: 3985
2026-04-28 11:46:06,730 INFO Loading BLIP-2 model: Salesforce/blip2-opt-2.7b on cuda
2026-04-28 11:46:06,883 INFO HTTP Request: GET https://huggingface.co/api/models/Salesforce/blip2-opt-2.7b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-28 11:46:06,944 INFO HTTP Request: HEAD https://huggingface.co/Salesforce/blip2-opt-2.7b/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 11:46:06,944 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-28 11:46:06,959 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/blip2-opt-2.7b/59a1ef6c1e5117b3f65523d1c6066825bcf315e3/processor_config.json "HTTP/1.1 200 OK"
2026-04-28 11:46:06,978 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/mo

In [5]:
# Upload captions.json to Kaggle dataset (run after Cell 4 finishes)

import os, json

# Verify output exists
captions_path = "/kaggle/working/captions.json"
if not os.path.exists(captions_path):
    print("❌ captions.json not found — did Cell 4 complete?")
else:
    with open(captions_path) as f:
        captions = json.load(f)
    print(f"✅ captions.json found — {len(captions):,} captions")

    # Sample check
    sample = list(captions.items())[:2]
    for item_id, caption in sample:
        print(f"\n  {item_id}: {caption}")

✅ captions.json found — 3,985 captions

  id_00000001: a woman wearing a white blouse and black shorts

  id_00000007: a woman wearing a tank top with the words snoop dogg on it


In [6]:
# Publish as Kaggle dataset

import json, os

USERNAME = "pankajdeopa"

# Write metadata
meta = {
    "title": "deepfashion-inshop-captions",
    "id": f"{USERNAME}/deepfashion-inshop-captions",
    "licenses": [{"name": "CC0-1.0"}]
}

# Create a clean folder with just captions.json
os.makedirs("/kaggle/working/captions-dataset", exist_ok=True)
!cp /kaggle/working/captions.json /kaggle/working/captions-dataset/captions.json

with open("/kaggle/working/captions-dataset/dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Metadata written. Uploading...")

# Upload
!kaggle datasets create -p /kaggle/working/captions-dataset/ --dir-mode zip

print(f"\n✅ Published at: https://www.kaggle.com/datasets/{USERNAME}/deepfashion-inshop-captions")

Metadata written. Uploading...
Dataset creation error: The requested title "deepfashion-inshop-captions" is already in use by a dataset. Please choose another title.

✅ Published at: https://www.kaggle.com/datasets/pankajdeopa/deepfashion-inshop-captions


In [7]:
import json
from pathlib import Path
import sys
sys.path.insert(0, "/kaggle/working/vision-search-engine")
import config

# Load image_paths.json
with open(config.IMAGE_PATHS_PATH) as f:
    image_paths = json.load(f)

print(f"Total entries in image_paths.json: {len(image_paths)}")

# Count unique item_ids
unique_items = {}
for k, v in image_paths.items():
    item_id = v["item_id"]
    if item_id not in unique_items:
        unique_items[item_id] = v["path"]

print(f"Unique item_ids: {len(unique_items)}")

# Check how many crops actually exist after remapping
marker = "/crops/"
existing = 0
missing = 0
for item_id, path in unique_items.items():
    idx = path.find(marker)
    if idx != -1:
        relative = path[idx + len(marker):]
        new_path = config.CROPS_DIR / relative
    else:
        new_path = Path(path)
    
    if new_path.exists():
        existing += 1
    else:
        missing += 1

print(f"\nCrops that exist:  {existing}")
print(f"Crops missing:     {missing}")
print(f"\nConclusion: BLIP-2 can only caption {existing} items")

Total entries in image_paths.json: 12612
Unique item_ids: 3985

Crops that exist:  3985
Crops missing:     0

Conclusion: BLIP-2 can only caption 3985 items
